# Prompt Swallowing — Spot Check

Compare three configurations:
1. **Trained student + prompt** (should be good — it had both advantages)
2. **Base Qwen3-0.6B + prompt** (baseline — no training, but full prompt)
3. **Trained student WITHOUT prompt** (the actual goal — does it behave as if the prompt were there?)

In [1]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

CHECKPOINT_DIR = "../outputs/prompt_swallowing/checkpoint-22000"
BASE_MODEL = "Qwen/Qwen3-0.6B"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# Load the tools config to build the system prompt
with open("../configs/prompt_swallowing_config.json") as f:
    config = json.load(f)

import sys; sys.path.insert(0, "..")
from src.data.data_generator import build_tool_system_prompt

SYSTEM_PROMPT = build_tool_system_prompt(config["tools"])
print(SYSTEM_PROMPT)

You are a helpful assistant with access to the following tools. Use them when appropriate to help answer the user's questions.

Available tools:
- calculator: Perform mathematical calculations
  Parameters:
    - expression (string, required): Mathematical expression to evaluate

- weather: Get weather information for a location
  Parameters:
    - location (string, required): Location to get weather for

- search: Search for information on the internet
  Parameters:
    - query (string, required): Search query

When you need to use a tool, respond with a JSON object in this format:
{"name": "tool_name", "parameters": {"param1": "value1"}}


In [3]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [4]:
# Load base model (untrained Qwen3-0.6B)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.eval()
print("Base model loaded")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Base model loaded


In [5]:
# Load trained student (base + LoRA adapter from checkpoint)
trained_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
trained_model = PeftModel.from_pretrained(trained_model, CHECKPOINT_DIR)
trained_model.eval()
print("Trained student loaded from", CHECKPOINT_DIR)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Trained student loaded from ../outputs/prompt_swallowing/checkpoint-22000


In [6]:
def generate_response(model, messages, max_new_tokens=256):
    """Generate a response given a list of chat messages."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    # Decode only the generated portion
    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

In [7]:
test_queries = [
    "What's 347 * 28?",
    "What's the weather like in Tokyo?",
    "Search for information about quantum computing",
    "Calculate 1024 / 16 + 55",
    "Check weather for Berlin",
]

## 1. Trained student WITH prompt

In [8]:
print("=" * 70)
print("TRAINED MODEL + SYSTEM PROMPT")
print("=" * 70)
for q in test_queries:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    response = generate_response(trained_model, messages)
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 70)

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


TRAINED MODEL + SYSTEM PROMPT

Q: What's 347 * 28?
A: <think>
Okay, the user is asking for the product of 347 and 28. Let me calculate that. I can use the calculator tool here. The expression would be 347 multiplied by 28. Let me do the math step by step. 347 times 20 is 6940, and 347 times 8 is 2776. Adding those together gives 6940 + 2776 = 9716. So the result should be 9716. I should present that clearly to the user.
</think>

{"result": "9716"}
----------------------------------------------------------------------

Q: What's the weather like in Tokyo?
A: <think>
Okay, the user is asking about the weather in Tokyo. I need to use the weather tool here. The parameters required are the location, which is Tokyo. I should make sure to format the response correctly with the location parameter. Let me check if there's any specific information needed, but since the user just wants the weather, the tool should handle that. I'll structure the JSON with the location set to "Tokyo".
</think>

{

## 2. Base Qwen3-0.6B WITH prompt (untrained baseline)

In [9]:
print("=" * 70)
print("BASE MODEL + SYSTEM PROMPT (no training)")
print("=" * 70)
for q in test_queries:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    response = generate_response(base_model, messages)
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 70)

BASE MODEL + SYSTEM PROMPT (no training)

Q: What's 347 * 28?
A: <think>
Okay, the user is asking for the product of 347 and 28. Let me check if I can use the calculator tool here. The calculator is the right tool for this. I need to make sure the expression is correctly formatted. The expression should be "347 * 28". Let me calculate that. 347 multiplied by 28... Hmm, 300*28 is 8400, 40*28 is 1120, 7*28 is 196. Adding those together: 8400 + 1120 is 9520, plus 196 is 9716. So the result should be 9716. I should present that as the answer.
</think>

{"name": "calculator", "parameters": {"expression": "347 * 28"}}
----------------------------------------------------------------------

Q: What's the weather like in Tokyo?
A: <think>
Okay, the user is asking about the weather in Tokyo. I need to use the weather tool here. The parameters required are the location, which is Tokyo. I should make sure to specify "Tokyo" in the location parameter. Let me check if there are any other parameters,

## 3. Trained student WITHOUT prompt (the actual test)

In [10]:
print("=" * 70)
print("TRAINED MODEL — NO SYSTEM PROMPT")
print("=" * 70)
for q in test_queries:
    messages = [
        {"role": "user", "content": q},
    ]
    response = generate_response(trained_model, messages)
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 70)

TRAINED MODEL — NO SYSTEM PROMPT

Q: What's 347 * 28?
A: <think>
Okay, the user is asking for the product of 347 and 28. Let me calculate that. First, I can break it down. 347 multiplied by 28. Maybe I can use the standard multiplication method. Let's see, 347 times 20 is 6940, and 347 times 8 is 2776. Adding those together: 6940 + 2776 equals 9716. Wait, is there another way to check? Let me verify. Alternatively, I can use the formula (300 + 47) * 28 = 300*28 + 47*28. 300*28 is 8400, and 47*28 is 1316. Adding those gives 8400 + 1316 = 9716. Yep, same result. So the answer should be 9716.
</think>

347 * 28 = 9716
----------------------------------------------------------------------

Q: What's the weather like in Tokyo?
A: <think>
Okay, the user is asking about the weather in Tokyo. I need to check if there's a specific weather service or database that can provide this information. Since I don't have a built-in weather API, I'll have to rely on my knowledge. I remember that Tokyo is 

## Side-by-side comparison

In [11]:
for q in test_queries:
    print("=" * 70)
    print(f"QUERY: {q}")
    print("=" * 70)

    msgs_with_prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    msgs_no_prompt = [
        {"role": "user", "content": q},
    ]

    r1 = generate_response(trained_model, msgs_with_prompt)
    r2 = generate_response(base_model, msgs_with_prompt)
    r3 = generate_response(trained_model, msgs_no_prompt)

    print(f"  [trained+prompt]   {r1[:200]}")
    print(f"  [base+prompt]      {r2[:200]}")
    print(f"  [trained-no-prompt] {r3[:200]}")
    print()

QUERY: What's 347 * 28?
  [trained+prompt]   <think>
Okay, the user is asking for the product of 347 and 28. Let me calculate that. I can use the calculator tool here. The expression would be 347 multiplied by 28. Let me do the math step by step
  [base+prompt]      <think>
Okay, the user is asking for the product of 347 and 28. Let me check if I can use the calculator tool here. The calculator is the right tool for this. I need to make sure the expression is cor
  [trained-no-prompt] <think>
Okay, the user is asking for the product of 347 and 28. Let me calculate that. First, I can break it down. 347 multiplied by 28. Maybe I can use the standard multiplication method. Let's see, 

QUERY: What's the weather like in Tokyo?
  [trained+prompt]   <think>
Okay, the user is asking about the weather in Tokyo. I need to use the weather tool here. The parameters required are the location, which is Tokyo. I should make sure to format the response co
  [base+prompt]      <think>
Okay, the user i